In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import proportional_hazard_test
from lifelines.utils import concordance_index
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats

In [ ]:
import os
import sys
from pathlib import Path

library_path = os.path.abspath('../src')
if library_path not in sys.path:
    sys.path.append(library_path)
library_path = Path(library_path)
library_path

In [ ]:
# load data
DATA_PATH = library_path.parent / "data"
PLOTS_PATH = library_path.parent / "plots"

cols_to_use = ["event", "months", "Sex", "Age", "Tumor", "sPCI", "pPCI", "CC"]

df = pd.read_csv(f"{DATA_PATH}/GPT_processed_survival_data.csv", usecols=cols_to_use)
df['months_round'] = df['months'].round().astype(int)

In [ ]:
t_cutoff = 57

df["months_trunc"] = df["months_round"].clip(upper=t_cutoff)
df["event_trunc"] = ((df["event"] == 1) & (df["months_round"] <= t_cutoff)).astype(int)

In [ ]:
df_base = df[["event_trunc", "months_trunc", "Sex", "Age", "Tumor", "CC"]].copy()
df_base.rename(columns={"event_trunc": "event", "months_trunc": "months"}, inplace=True)

df_S = df[["event_trunc", "months_trunc", "Sex", "Age", "Tumor","CC", "sPCI"]].copy()
df_S.rename(columns={"event_trunc": "event", "months_trunc": "months"}, inplace=True)

df_P = df[["event_trunc", "months_trunc", "Sex", "Age", "Tumor", "CC", "pPCI"]].copy()
df_P.rename(columns={"event_trunc": "event", "months_trunc": "months"}, inplace=True)

In [ ]:
# ── 1. Data Overview ──────────────────────────────────────────────────────────
print(f"Total observations: {len(df)}")

print("\nMissing values:")
display(df[["Sex", "Age", "Tumor", "sPCI", "pPCI", "CC", "event", "months"]].isnull().sum().rename("n_missing").to_frame().T)

print(f"\nSurvival (truncated at {t_cutoff} months):")
n_events = df["event_trunc"].sum()
print(f"  Events  : {n_events} / {len(df)} ({100 * n_events / len(df):.1f}%)")
print(f"  Median follow-up : {df['months_trunc'].median():.1f} months")
print(f"  Mean follow-up   : {df['months_trunc'].mean():.1f} months")

print("\nPCI missingness:")
both    = (df["sPCI"].notna() & df["pPCI"].notna()).sum()
only_s  = (df["sPCI"].notna() & df["pPCI"].isna()).sum()
only_p  = (df["sPCI"].isna()  & df["pPCI"].notna()).sum()
neither = (df["sPCI"].isna()  & df["pPCI"].isna()).sum()
print(f"  Both: {both}  |  sPCI only: {only_s}  |  pPCI only: {only_p}  |  Neither: {neither}")

print("\nPCI distribution:")
display(df[["sPCI", "pPCI"]].describe().round(2))

print("\nTumor type counts:")
display(df["Tumor"].value_counts().sort_index().rename("n").to_frame().T)

print("\nCC distribution:")
display(df["CC"].value_counts().sort_index().rename("n").to_frame().T)


In [ ]:
# ── 2. Preprocessing: Tumor Dummies & Sex Recode ─────────────────────────────
# Sex: 1 → 0 (female), 2 → 1 (male)
# Tumor: one-hot encoded, Tumor type 1 (most frequent) as reference category

def encode_covariates(raw_df: pd.DataFrame, pci_col: str = None) -> pd.DataFrame:
    """Recode Sex (0/1) and one-hot-encode Tumor. Drop rows missing pci_col."""
    d        = raw_df.copy()
    d["Sex"] = d["Sex"] - 1
    dummies  = pd.get_dummies(d["Tumor"], prefix="T", drop_first=False, dtype=int)
    ref_col  = "T_1"
    if ref_col in dummies.columns:
        dummies = dummies.drop(columns=[ref_col])   # Tumor 1 = reference
    d = pd.concat([d.drop(columns=["Tumor"]), dummies], axis=1)
    if pci_col is not None:
        d = d.dropna(subset=[pci_col])
    return d

df_base = encode_covariates(df_base)
df_S    = encode_covariates(df_S, pci_col="sPCI")
df_P    = encode_covariates(df_P, pci_col="pPCI")

tumor_cols  = sorted(c for c in df_base.columns if c.startswith("T_"))
base_covars = ["Age", "Sex", "CC"] + tumor_cols

print(f"{'Dataset':<18} {'N':>5}  {'Events':>7}")
print("-" * 34)
for label, d in [("df_base", df_base), ("df_S  (sPCI)", df_S), ("df_P  (pPCI)", df_P)]:
    print(f"{label:<18} {len(d):>5}  {int(d['event'].sum()):>7}")
print(f"\nBase covariates: {base_covars}")


In [ ]:
# ── 3. Cox Proportional Hazards Models (Sex stratified) ──────────────────────
# Sex violates the PH assumption (Schoenfeld test) → stratify rather than
# include as a linear covariate. This allows each sex its own baseline hazard
# while estimating a shared PCI / Tumor / Age effect.

cph_base = CoxPHFitter()
cph_S    = CoxPHFitter()
cph_P    = CoxPHFitter()

cph_base.fit(df_base, duration_col="months", event_col="event", strata=["Sex"])
cph_S   .fit(df_S,    duration_col="months", event_col="event", strata=["Sex"])
cph_P   .fit(df_P,    duration_col="months", event_col="event", strata=["Sex"])

print("=" * 60)
print("BASE MODEL  (Age + Tumor + CC — Sex stratified)")
print("=" * 60)
cph_base.print_summary(decimals=3)

print("\n" + "=" * 60)
print("MODEL A — Surgical PCI (sPCI, Sex stratified)")
print("=" * 60)
cph_S.print_summary(decimals=3)

print("\n" + "=" * 60)
print("MODEL B — Pathological PCI (pPCI, Sex stratified)")
print("=" * 60)
cph_P.print_summary(decimals=3)


In [ ]:
# ── 4. Proportional Hazards Assumption (Schoenfeld residuals, rank transform) ─
print("── BASE ──────────────────────────────────────────────────────────────────")
proportional_hazard_test(cph_base, df_base, time_transform="rank").print_summary(decimals=3)

print("\n── MODEL A (sPCI) ────────────────────────────────────────────────────────")
proportional_hazard_test(cph_S, df_S, time_transform="rank").print_summary(decimals=3)

print("\n── MODEL B (pPCI) ────────────────────────────────────────────────────────")
proportional_hazard_test(cph_P, df_P, time_transform="rank").print_summary(decimals=3)


In [ ]:
# ── 5. Model Comparison Table ────────────────────────────────────────────────
# LRT requires base model re-fitted on the same complete-case rows as each
# augmented model (different N when sPCI / pPCI have missing data).
from scipy.stats import chi2

idx_S      = df_S.index.intersection(df_base.index)
idx_P      = df_P.index.intersection(df_base.index)

cph_base_S = CoxPHFitter().fit(df_base.loc[idx_S], duration_col="months", event_col="event")
cph_base_P = CoxPHFitter().fit(df_base.loc[idx_P], duration_col="months", event_col="event")

lrt_A = 2 * (cph_S.log_likelihood_ - cph_base_S.log_likelihood_)
lrt_B = 2 * (cph_P.log_likelihood_ - cph_base_P.log_likelihood_)
p_A   = chi2.sf(lrt_A, df=1)
p_B   = chi2.sf(lrt_B, df=1)

aic = lambda cph: -2 * cph.log_likelihood_ + 2 * len(cph.params_)

rows = {
    "Base"           : {"N": len(df_base), "Events": int(df_base["event"].sum()),
                        "C-index": cph_base.concordance_index_, "AIC": aic(cph_base),
                        "LRT χ²": np.nan, "LRT p": np.nan,
                        "PCI HR": np.nan, "PCI p": np.nan},
    "Model A (sPCI)" : {"N": len(df_S), "Events": int(df_S["event"].sum()),
                        "C-index": cph_S.concordance_index_, "AIC": aic(cph_S),
                        "LRT χ²": lrt_A, "LRT p": p_A,
                        "PCI HR": np.exp(cph_S.params_["sPCI"]),
                        "PCI p":  cph_S.summary.loc["sPCI", "p"]},
    "Model B (pPCI)" : {"N": len(df_P), "Events": int(df_P["event"].sum()),
                        "C-index": cph_P.concordance_index_, "AIC": aic(cph_P),
                        "LRT χ²": lrt_B, "LRT p": p_B,
                        "PCI HR": np.exp(cph_P.params_["pPCI"]),
                        "PCI p":  cph_P.summary.loc["pPCI", "p"]},
}
comparison = pd.DataFrame(rows).T.round(4)
display(comparison)

print(f"\nΔC  (B − A)  = {cph_P.concordance_index_ - cph_S.concordance_index_:+.4f}")
print(f"ΔAIC (A − B) = {aic(cph_S) - aic(cph_P):+.2f}  (negative = Model B preferred by AIC)")


In [ ]:
# ── 6. Bootstrap ΔC-index: paired non-nested model comparison ─────────────────
# LRT is invalid for non-nested models (A vs B).
# Paired bootstrap on shared rows gives a 95% CI for ΔC = C(B) − C(A).

common_idx = df_S.index.intersection(df_P.index)
df_S_cc    = df_S.loc[common_idx].reset_index(drop=True)
df_P_cc    = df_P.loc[common_idx].reset_index(drop=True)

np.random.seed(42)
N_BOOT       = 1000
delta_c_boot = []

for _ in range(N_BOOT):
    idx = np.random.choice(len(df_S_cc), size=len(df_S_cc), replace=True)
    b_S = df_S_cc.iloc[idx].reset_index(drop=True)
    b_P = df_P_cc.iloc[idx].reset_index(drop=True)
    try:
        c_S = CoxPHFitter().fit(b_S, duration_col="months", event_col="event").concordance_index_
        c_P = CoxPHFitter().fit(b_P, duration_col="months", event_col="event").concordance_index_
        delta_c_boot.append(c_P - c_S)
    except Exception:
        continue

delta_c_boot = np.array(delta_c_boot)
obs_delta    = cph_P.concordance_index_ - cph_S.concordance_index_
ci_lo, ci_hi = np.percentile(delta_c_boot, [2.5, 97.5])

print("Bootstrap C-index comparison  (Model B pPCI − Model A sPCI)")
print(f"  Common N         = {len(df_S_cc)}")
print(f"  Successful runs  = {len(delta_c_boot)} / {N_BOOT}")
print(f"  Observed ΔC      = {obs_delta:+.4f}")
print(f"  Bootstrap 95% CI = [{ci_lo:+.4f}, {ci_hi:+.4f}]")
if   ci_lo > 0: print("  → pPCI discriminates significantly better.")
elif ci_hi < 0: print("  → sPCI discriminates significantly better.")
else:           print("  → No significant difference in discrimination (CI crosses 0).")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(delta_c_boot, bins=40, color="steelblue", edgecolor="white", alpha=0.85)
ax.axvline(obs_delta, color="crimson", lw=2,   label=f"Observed ΔC = {obs_delta:+.4f}")
ax.axvline(ci_lo,     color="dimgray", lw=1.5, ls="--",
           label=f"95% CI [{ci_lo:+.4f}, {ci_hi:+.4f}]")
ax.axvline(ci_hi,     color="dimgray", lw=1.5, ls="--")
ax.axvline(0,         color="black",   lw=1,   ls=":")
ax.set_xlabel("ΔC  (Model B − Model A)")
ax.set_ylabel("Bootstrap frequency")
ax.set_title(f"Bootstrap ΔC-index  (n = {len(delta_c_boot)} resamples)")
ax.legend()
plt.tight_layout()
plt.savefig(PLOTS_PATH / "cox_bootstrap_delta_c.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── 7. Forest Plots ───────────────────────────────────────────────────────────
def cox_forest_plot(cph, title, ax, highlight_var=None):
    """HR ± 95% CI for each covariate; highlight_var shown in red."""
    smry     = cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%"]].copy()
    rows_rev = list(smry.iloc[::-1].iterrows())   # bottom-to-top order

    for i, (var, row) in enumerate(rows_rev):
        is_hi  = (var == highlight_var)
        color  = "crimson" if is_hi else "steelblue"
        ms     = 9 if is_hi else 6
        lw     = 2.0 if is_hi else 1.2
        xerr   = [[row["exp(coef)"] - row["exp(coef) lower 95%"]],
                  [row["exp(coef) upper 95%"] - row["exp(coef)"]]]
        ax.errorbar(row["exp(coef)"], i, xerr=xerr,
                    fmt="o", color=color, capsize=4, markersize=ms, linewidth=lw)

    ax.set_yticks(range(len(rows_rev)))
    ax.set_yticklabels([v for v, _ in rows_rev], fontsize=8)
    ax.axvline(1, color="black", lw=1, ls="--", alpha=0.5)
    ax.set_xscale("log")
    ax.set_xlabel("Hazard Ratio (95% CI, log scale)")
    ax.set_title(title, fontsize=11)
    ax.grid(axis="x", alpha=0.25)


fig, axes = plt.subplots(1, 2, figsize=(15, 6))
cox_forest_plot(cph_S, "Model A — Surgical PCI (sPCI)",     axes[0], highlight_var="sPCI")
cox_forest_plot(cph_P, "Model B — Pathological PCI (pPCI)", axes[1], highlight_var="pPCI")

plt.suptitle("Cox PH Models — Forest Plots", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(PLOTS_PATH / "cox_forest_plots.png", dpi=150, bbox_inches="tight")
plt.show()
